In [1]:
import pandas as pd
import numpy as np
from datetime import timedelta

np.random.seed(42)

In [6]:
# Agent decisions (output of notebook 07)
agent_actions = pd.read_csv("../outputs/agent_actions.csv")

# Orders and customers to reconstruct timelines
orders = pd.read_csv("../olist/olist_orders_dataset.csv")
customers = pd.read_csv("../olist/olist_customers_dataset.csv")


In [7]:
# Map customer_id → customer_unique_id
orders = orders.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left"
)

orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6


In [8]:
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

orders = orders.sort_values("order_purchase_timestamp")

In [10]:
orders_with_uid[["customer_id", "customer_unique_id"]].head()

,customer_id,customer_unique_id
0,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff
1,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231
2,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8
3,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977
4,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6


In [9]:
# Define dataset start and end
start_date = orders["order_purchase_timestamp"].min()
end_date = orders["order_purchase_timestamp"].max()

# Split into 3 sequential windows
orders["time_window"] = pd.qcut(
    orders["order_purchase_timestamp"],
    q=3,
    labels=["W1", "W2", "W3"]
)

orders[["order_purchase_timestamp", "time_window"]].head()

,order_purchase_timestamp,time_window
4541,2016-09-04 21:15:19,W1
4396,2016-09-05 00:15:34,W1
10071,2016-09-13 15:24:19,W1
30710,2016-09-15 12:16:38,W1
83078,2016-10-02 22:07:52,W1


In [10]:
agent_actions = agent_actions.merge(
    orders[["customer_unique_id", "time_window"]],
    on="customer_unique_id",
    how="left"
)

agent_actions.head()

,customer_unique_id,persona,agent_action,time_window
0,0000366f3b9a7992bf8c76cfdf3221e2,Silent Loyal Customers,No Action,W3
1,0000f6ccb0745a6a4b88665a16c9f078,High-Value Enthusiasts,No Action,W1
2,00053a61a98854899e70ed204dd4bafe,High-Value Enthusiasts,No Action,W2
3,0005ef4cd20d2893f0d9fbd94d3c0d97,Silent Loyal Customers,No Action,W2
4,0006fdc98a402fceb4eb0ee528f6a8d4,Silent Loyal Customers,No Action,W1


In [11]:
feedback_outcomes = [
    "positive_response",   # purchase again / improved sentiment
    "no_response",         # nothing happened
    "negative_response",   # complaint / ignored
    "churned"              # no future activity
]

feedback_probs = {
    "No Action": [0.05, 0.8, 0.05, 0.1],
    "Personalized Discount Offer": [0.35, 0.4, 0.15, 0.1],
    "Loyalty Points + Feedback Request": [0.45, 0.4, 0.1, 0.05],
    "Win-Back Campaign": [0.25, 0.35, 0.15, 0.25],
}

In [12]:
def simulate_feedback(action):
    probs = feedback_probs[action]
    return np.random.choice(feedback_outcomes, p=probs)

agent_actions["feedback_outcome"] = agent_actions["agent_action"].apply(
    simulate_feedback
)

agent_actions.head()

,customer_unique_id,persona,agent_action,time_window,feedback_outcome
0,0000366f3b9a7992bf8c76cfdf3221e2,Silent Loyal Customers,No Action,W3,no_response
1,0000f6ccb0745a6a4b88665a16c9f078,High-Value Enthusiasts,No Action,W1,churned
2,00053a61a98854899e70ed204dd4bafe,High-Value Enthusiasts,No Action,W2,no_response
3,0005ef4cd20d2893f0d9fbd94d3c0d97,Silent Loyal Customers,No Action,W2,no_response
4,0006fdc98a402fceb4eb0ee528f6a8d4,Silent Loyal Customers,No Action,W1,no_response


In [13]:
feedback_score_map = {
    "positive_response": 1,
    "no_response": 0,
    "negative_response": -0.5,
    "churned": -1
}

agent_actions["feedback_score"] = agent_actions["feedback_outcome"].map(
    feedback_score_map
)

In [14]:
feedback_history = (
    agent_actions
    .groupby("customer_unique_id")
    .agg(
        total_feedback_score=("feedback_score", "sum"),
        num_actions=("agent_action", "count"),
        num_positive=("feedback_outcome", lambda x: (x == "positive_response").sum()),
        num_negative=("feedback_outcome", lambda x: (x == "negative_response").sum()),
        num_churn=("feedback_outcome", lambda x: (x == "churned").sum())
    )
    .reset_index()
)

feedback_history.head()


,customer_unique_id,total_feedback_score,num_actions,num_positive,num_negative,num_churn
0,0000366f3b9a7992bf8c76cfdf3221e2,0.0,1,0,0,0
1,0000f6ccb0745a6a4b88665a16c9f078,-1.0,1,0,0,1
2,00053a61a98854899e70ed204dd4bafe,0.0,1,0,0,0
3,0005ef4cd20d2893f0d9fbd94d3c0d97,0.0,1,0,0,0
4,0006fdc98a402fceb4eb0ee528f6a8d4,0.0,1,0,0,0


In [15]:
agent_feedback = agent_actions.merge(
    feedback_history,
    on="customer_unique_id",
    how="left"
)

agent_feedback.head()


,customer_unique_id,persona,agent_action,time_window,feedback_outcome,feedback_score,total_feedback_score,num_actions,num_positive,num_negative,num_churn
0,0000366f3b9a7992bf8c76cfdf3221e2,Silent Loyal Customers,No Action,W3,no_response,0.0,0.0,1,0,0,0
1,0000f6ccb0745a6a4b88665a16c9f078,High-Value Enthusiasts,No Action,W1,churned,-1.0,-1.0,1,0,0,1
2,00053a61a98854899e70ed204dd4bafe,High-Value Enthusiasts,No Action,W2,no_response,0.0,0.0,1,0,0,0
3,0005ef4cd20d2893f0d9fbd94d3c0d97,Silent Loyal Customers,No Action,W2,no_response,0.0,0.0,1,0,0,0
4,0006fdc98a402fceb4eb0ee528f6a8d4,Silent Loyal Customers,No Action,W1,no_response,0.0,0.0,1,0,0,0


In [16]:
agent_feedback.to_csv(
    "../outputs/agent_feedback_history.csv",
    index=False
)

print("agent_feedback_history.csv saved successfully.")


agent_feedback_history.csv saved successfully.


In [17]:
import pandas as pd

feedback = pd.read_csv("../outputs/agent_feedback_history.csv")
feedback.head()


,customer_unique_id,persona,agent_action,time_window,feedback_outcome,feedback_score,total_feedback_score,num_actions,num_positive,num_negative,num_churn
0,0000366f3b9a7992bf8c76cfdf3221e2,Silent Loyal Customers,No Action,W3,no_response,0.0,0.0,1,0,0,0
1,0000f6ccb0745a6a4b88665a16c9f078,High-Value Enthusiasts,No Action,W1,churned,-1.0,-1.0,1,0,0,1
2,00053a61a98854899e70ed204dd4bafe,High-Value Enthusiasts,No Action,W2,no_response,0.0,0.0,1,0,0,0
3,0005ef4cd20d2893f0d9fbd94d3c0d97,Silent Loyal Customers,No Action,W2,no_response,0.0,0.0,1,0,0,0
4,0006fdc98a402fceb4eb0ee528f6a8d4,Silent Loyal Customers,No Action,W1,no_response,0.0,0.0,1,0,0,0


In [18]:
feedback_outcome_counts = (
    feedback["feedback_outcome"]
    .value_counts()
    .reset_index()
)

feedback_outcome_counts.columns = ["feedback_outcome", "count"]
feedback_outcome_counts

,feedback_outcome,count
0,no_response,30269
1,churned,4263
2,positive_response,3885
3,negative_response,2662


In [19]:
feedback_by_window = (
    feedback
    .groupby(["time_window", "feedback_outcome"])
    .size()
    .reset_index(name="count")
    .sort_values(["time_window", "count"], ascending=[True, False])
)

feedback_by_window

,time_window,feedback_outcome,count
2,W1,no_response,9476
3,W1,positive_response,1940
0,W1,churned,1516
1,W1,negative_response,1117
6,W2,no_response,10274
4,W2,churned,1389
7,W2,positive_response,1295
5,W2,negative_response,916
10,W3,no_response,10519
8,W3,churned,1358


In [21]:
feedback_pivot = pd.pivot_table(
    feedback,
    index="time_window",
    columns="feedback_outcome",
    values="customer_unique_id",
    aggfunc="count",
    fill_value=0
)

feedback_pivot

feedback_outcome,churned,negative_response,no_response,positive_response
time_window,,,,
W1,1516,1117,9476,1940
W2,1389,916,10274,1295
W3,1358,629,10519,650


In [22]:
action_feedback = (
    feedback
    .groupby(["agent_action", "feedback_outcome"])
    .size()
    .reset_index(name="count")
    .sort_values(["agent_action", "count"], ascending=False)
)

action_feedback

,agent_action,feedback_outcome,count
14,Win-Back Campaign,no_response,344
15,Win-Back Campaign,positive_response,249
12,Win-Back Campaign,churned,239
13,Win-Back Campaign,negative_response,148
10,Personalized Discount Offer,no_response,2207
11,Personalized Discount Offer,positive_response,1893
9,Personalized Discount Offer,negative_response,808
8,Personalized Discount Offer,churned,510
6,No Action,no_response,27691
4,No Action,churned,3509
